In [1]:
import os

print(os.getcwd())

import pandas as pd

# =====================================================
# LOAD DATASETS
# =====================================================

payroll = pd.read_csv("NBA_team_payroll_2026.csv")
salarycap = pd.read_csv("NBA_team_salarycap_2026.csv")
stats = pd.read_csv("NBA_team_stats.csv")
advanced = pd.read_csv("NBA_team_stats2.csv")

# =====================================================
# CLEAN TEAM NAMES
# =====================================================

# Remove asterisks from Basketball Reference playoff markers
stats["Team"] = stats["Team"].str.replace("*", "", regex=False)
advanced["Team"] = advanced["Team"].str.replace("*", "", regex=False)

# =====================================================
# CREATE TEAM NAME MAPPING
# =====================================================

team_map = {
    "Atlanta Hawks": "ATL",
    "Boston Celtics": "BOS",
    "Brooklyn Nets": "BRK",
    "Charlotte Hornets": "CHO",
    "Chicago Bulls": "CHI",
    "Cleveland Cavaliers": "CLE",
    "Dallas Mavericks": "DAL",
    "Denver Nuggets": "DEN",
    "Detroit Pistons": "DET",
    "Golden State Warriors": "GSW",
    "Houston Rockets": "HOU",
    "Indiana Pacers": "IND",
    "Los Angeles Clippers": "LAC",
    "Los Angeles Lakers": "LAL",
    "Memphis Grizzlies": "MEM",
    "Miami Heat": "MIA",
    "Milwaukee Bucks": "MIL",
    "Minnesota Timberwolves": "MIN",
    "New Orleans Pelicans": "NOP",
    "New York Knicks": "NYK",
    "Oklahoma City Thunder": "OKC",
    "Orlando Magic": "ORL",
    "Philadelphia 76ers": "PHI",
    "Phoenix Suns": "PHO",
    "Portland Trail Blazers": "POR",
    "Sacramento Kings": "SAC",
    "San Antonio Spurs": "SAS",
    "Toronto Raptors": "TOR",
    "Utah Jazz": "UTA",
    "Washington Wizards": "WAS"
}

stats["Team"] = stats["Team"].map(team_map)
advanced["Team"] = advanced["Team"].map(team_map)

# Remove any rows that aren't actual NBA teams
stats = stats[stats["Team"].notna()]
advanced = advanced[advanced["Team"].notna()]

# =====================================================
# SELECT IMPORTANT COLUMNS
# =====================================================

payroll = payroll[
    [
        "Team",
        "Avg Age",
        "Total Cash",
        "Dead"
    ]
].rename(
    columns={
        "Avg Age": "AvgAge",
        "Total Cash": "Payroll",
        "Dead": "DeadCash"
    }
)

salarycap = salarycap[
    [
        "Team",
        "Team Total Cap Allocations",
        "Cap Space",
        "Dead Cap"
    ]
].rename(
    columns={
        "Team Total Cap Allocations": "CapAllocations",
        "Dead Cap": "DeadCap"
    }
)

stats = stats[
    [
        "Team",
        "PTS",
        "FG%",
        "3P%",
        "FT%",
        "TRB",
        "AST",
        "STL",
        "BLK",
        "TOV"
    ]
]

advanced = advanced[
    [
        "Team",
        "Age",
        "W",
        "L",
        "MOV",
        "SRS",
        "ORtg",
        "DRtg",
        "NRtg",
        "Pace",
        "TS%"
    ]
].rename(
    columns={
        "Age": "RosterAge"
    }
)

# =====================================================
# MERGE DATASETS
# =====================================================

df = payroll.merge(salarycap, on="Team", how="inner")

df = df.merge(stats, on="Team", how="inner")

df = df.merge(advanced, on="Team", how="inner")

# =====================================================
# FEATURE ENGINEERING
# =====================================================

df["WinPct"] = df["W"] / (df["W"] + df["L"])

df["CostPerWin"] = df["Payroll"] / df["W"]

df["PayrollRank"] = df["Payroll"].rank(
    ascending=False,
    method="min"
)

df["CapAllocationRank"] = df["CapAllocations"].rank(
    ascending=False,
    method="min"
)

df["PointDiffPerDollar"] = (
    df["MOV"] / (df["Payroll"] / 1_000_000)
)

df["NetRatingPerMillion"] = (
    df["NRtg"] / (df["Payroll"] / 1_000_000)
)

# =====================================================
# SORT BY WINS
# =====================================================

df = df.sort_values(
    by="W",
    ascending=False
)

# =====================================================
# ROUND DECIMAL COLUMNS
# =====================================================

round_cols = [
    "WinPct",
    "CostPerWin",
    "PointDiffPerDollar",
    "NetRatingPerMillion"
]

df[round_cols] = df[round_cols].round(3)

# =====================================================
# SAVE DATASET
# =====================================================

df.to_csv(
    "NBA_Master_Dataset_2025_26.csv",
    index=False
)

print("Dataset created successfully!")
print(df.head())

print("\nRows:", len(df))
print("Columns:", len(df.columns))

c:\Users\mahab\OneDrive\Desktop\Summer2026python\summer-2026-python
Dataset created successfully!
   Team  AvgAge    Payroll  DeadCash  CapAllocations  Cap Space  DeadCap  \
18  OKC    24.6  187827634   2298085       188057857  -33410857  2296274   
26  SAS    26.6  187078708   7569181       183990669  -29343669  7258201   
14  DET    25.9  179870831   6210275       193082507  -38435507  9337154   
7   BOS    26.1  196926936    582842       194526296  -39879296   469063   
22  DEN    26.7  192006289    636435       200743895  -46096895        0   

      PTS    FG%    3P%  ...   DRtg  NRtg  Pace    TS%  WinPct   CostPerWin  \
18  119.0  0.484  0.365  ...  107.7  11.2  99.3  0.599   0.780  2934806.781   
26  119.8  0.483  0.359  ...  111.3   8.3  99.9  0.595   0.756  3017398.516   
14  117.8  0.485  0.356  ...  109.7   8.2  99.3  0.583   0.732  2997847.183   
7   114.9  0.467  0.367  ...  112.7   8.1  94.8  0.583   0.683  3516552.429   
22  122.1  0.496  0.396  ...  117.4   5.2  98.4  0

In [2]:
df.to_csv(
    "NBA_Master_Dataset_2026.csv",
    index=False
)

In [3]:
import pandas as pd
df = pd.read_csv('NBA_Master_Dataset_2026.csv')

In [4]:
df.head()

,Team,AvgAge,Payroll,DeadCash,CapAllocations,Cap Space,DeadCap,PTS,FG%,3P%,...,DRtg,NRtg,Pace,TS%,WinPct,CostPerWin,PayrollRank,CapAllocationRank,PointDiffPerDollar,NetRatingPerMillion
0,OKC,24.6,187827634,2298085,188057857,-33410857,2296274,119.0,0.484,0.365,...,107.7,11.2,99.3,0.599,0.780,2934806.781,18.0,22.0,0.059,0.060
1,SAS,26.6,187078708,7569181,183990669,-29343669,7258201,119.8,0.483,0.359,...,111.3,8.3,99.9,0.595,0.756,3017398.516,20.0,24.0,0.044,0.044
2,DET,25.9,179870831,6210275,193082507,-38435507,9337154,117.8,0.485,0.356,...,109.7,8.2,99.3,0.583,0.732,2997847.183,24.0,20.0,0.045,0.046
3,BOS,26.1,196926936,582842,194526296,-39879296,469063,114.9,0.467,0.367,...,112.7,8.1,94.8,0.583,0.683,3516552.429,10.0,17.0,0.039,0.041
4,DEN,26.7,192006289,636435,200743895,-46096895,0,122.1,0.496,0.396,...,117.4,5.2,98.4,0.616,0.659,3555672.019,12.0,10.0,0.027,0.027


In [ ]:
df.columns

Index(['Team', 'AvgAge', 'Payroll', 'DeadCash', 'CapAllocations', 'Cap Space',
       'DeadCap', 'PTS', 'FG%', '3P%', 'FT%', 'TRB', 'AST', 'STL', 'BLK',
       'TOV', 'RosterAge', 'W', 'L', 'MOV', 'SRS', 'ORtg', 'DRtg', 'NRtg',
       'Pace', 'TS%', 'WinPct', 'CostPerWin', 'PayrollRank',
       'CapAllocationRank', 'PointDiffPerDollar', 'NetRatingPerMillion'],
      dtype='str')

In [6]:
# Decision tree example: predict team wins from points, assists, and rebounds
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load the team dataset
path = "NBA_Master_Dataset_2026.csv"
df = pd.read_csv(path)

# Select features and target
X = df[["PTS", "AST", "TRB"]]
y = df["W"]

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train DecisionTreeRegressor
model = DecisionTreeRegressor(random_state=42)
model.fit(X_train, y_train)

# Predict and evaluate
preds = model.predict(X_test)
mae = mean_absolute_error(y_test, preds)
rmse = mean_squared_error(y_test, preds) ** 0.5
r2 = r2_score(y_test, preds)

print("DecisionTreeRegressor to predict team wins")
print(f"Test samples: {len(y_test)}")
print(f"MAE: {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R2: {r2:.3f}")

print("Feature importances:")
for name, importance in zip(X.columns, model.feature_importances_):
    print(f"  {name}: {importance:.4f}")

DecisionTreeRegressor to predict team wins
Test samples: 6
MAE: 8.500
RMSE: 9.925
R2: 0.226
Feature importances:
  PTS: 0.3136
  AST: 0.0918
  TRB: 0.5946


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

if "model" not in globals():

    df = pd.read_csv("NBA_Master_Dataset_2026.csv")
    X = df[["PTS", "AST", "TRB"]]
    y = df["W"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = DecisionTreeRegressor(random_state=42)
    model.fit(X_train, y_train)

prediction = model.predict([[110, 25, 45]])
print("Predicted Wins:", prediction[0])
print("Predicted Wins:", prediction[0])

NameError: name 'model' is not defined